# Анализ пролонгаций договоров — 2023 год

**Цель:** рассчитать коэффициенты пролонгации для каждого аккаунт-менеджера и отдела в целом за каждый месяц и за год.

**Два коэффициента:**
- **К1 (первый месяц)** — доля отгрузки проектов, пролонгированных в первый месяц после завершения
- **К2 (второй месяц)** — доля отгрузки проектов, пролонгированных во второй месяц (среди тех, что не пролонгировались в первый)

**Период расчёта:** декабрь 2022 — январь 2024. Декабрь 2022 и январь 2024 включены как периоды наблюдения: без них проекты, завершившиеся в ноябре 2022 и декабре 2023, не попали бы ни в один расчёт K1.

In [ ]:
import sys, os

if "google.colab" in sys.modules:
    # Colab: клонируем репозиторий, чтобы модули были доступны для импорта
    repo_path = "/content/prolongation-analysis"
    if not os.path.exists(repo_path):
        os.system(f"git clone https://github.com/RezPoint/prolongation-analysis.git {repo_path}")
    os.chdir(repo_path)
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)
else:
    # Локально: добавляем папку ноутбука в sys.path
    notebook_dir = os.path.dirname(os.path.abspath('prolongation_analysis.ipynb'))
    if notebook_dir not in sys.path:
        sys.path.insert(0, notebook_dir)

In [ ]:
import subprocess, sys

try:
    import xlsxwriter
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xlsxwriter", "-q"])
    import xlsxwriter

print("xlsxwriter", xlsxwriter.__version__)

## 1. Загрузка и подготовка данных

In [ ]:
from data         import load_data, get_month_cols, clean_financial, deduplicate_prolongations, build_project_df
from calculations import build_col_index

# Загружаем два источника: таблицу пролонгаций и финансовые данные
prol, fin  = load_data()

# Определяем столбцы с месяцами (всё кроме id, Account, Причина дубля)
month_cols = get_month_cols(fin)

# Приводим значения к числам: 'стоп', 'в ноль', NaN → 0; агрегируем дубли по id
fin_agg    = clean_financial(fin, month_cols)

# Убираем дубли в пролонгациях: один проект — одна запись last_month
prol       = deduplicate_prolongations(prol)

# Объединяем пролонгации с финансовыми данными по id
df         = build_project_df(prol, fin_agg, month_cols)

# Обратный индекс: название месяца → порядковый номер (для сдвигов M-1, M-2)
col_to_idx = build_col_index(month_cols)

print('prolongations:', len(prol), 'строк')
print('financial_data (после агрегации):', len(fin_agg), 'строк')
print('Итоговый датафрейм:', len(df), 'проектов')
df.head(3)

### Качество данных: расхождения между prolongations.csv и financial_data.csv

Среди всех проектов выявлено три типа расхождений:

| Тип | Описание | Влияние на расчёт |
|-----|----------|-------------------|
| **Нет выручки вообще** | Проект не имеет отгрузки ни в одном месяце | Исключается из знаменателя (), не влияет на К1/К2 |
| **Выручка закончилась раньше** | last_month указан позже фактического | Аналогично — исключается фильтром, не влияет |
| **Выручка продолжается ПОСЛЕ last_month** | Проект якобы завершён, но деньги идут дальше | Потенциальная пролонгация, не отражённая в prolongations.csv |

Последний тип наиболее критичен: такие проекты могут быть фактически пролонгированы, но не попадают в числитель К1/К2.  
**Вывод:** расхождения являются проблемой качества исходных данных. В соответствии с ТЗ prioritет у prolongations.csv — код работает корректно, однако реальные коэффициенты могут быть выше расчётных.

In [ ]:
# Контроль качества данных
# Проверяем соответствие last_month в prolongations.csv и фактических финансовых данных

problems = []
for _, row in df.iterrows():
    lm = row['last_month_col']
    if lm not in month_cols or row[lm] != 0:
        continue
    actual_last = next((m for m in reversed(month_cols) if row[m] > 0), None)
    lm_idx = month_cols.index(lm)
    actual_idx = month_cols.index(actual_last) if actual_last else None

    if actual_last is None:
        cat = 'нет выручки вообще'
    elif actual_idx < lm_idx:
        cat = 'выручка закончилась раньше last_month'
    else:
        cat = 'выручка продолжается ПОСЛЕ last_month'

    problems.append({
        'id': row['id'], 'AM': row['AM'],
        'last_month (CSV)': lm,
        'фактический последний месяц': actual_last,
        'тип расхождения': cat
    })

df_problems = pd.DataFrame(problems)
print(f'Проектов с расхождением: {len(df_problems)} из {len(df)} ({len(df_problems)/len(df):.0%})')
print()
print('Сводка по типам:')
print(df_problems['тип расхождения'].value_counts().to_string())
display(df_problems)

## 2. Расчёт коэффициентов K1 и K2

### Логика расчёта для месяца M:

**К1 (первый месяц):**
- Берём все проекты с `last_month = M-1` и ненулевой отгрузкой в M-1
- Знаменатель = сумма отгрузки этих проектов за M-1
- Числитель = сумма отгрузки за M тех из них, у кого отгрузка в M > 0
- К1 = Числитель / Знаменатель

**К2 (второй месяц):**
- Берём проекты с `last_month = M-2`, ненулевой отгрузкой в M-2 и **без** отгрузки в M-1
- Знаменатель = сумма отгрузки этих проектов за M-2
- Числитель = сумма отгрузки за M тех из них, у кого отгрузка в M > 0
- К2 = Числитель / Знаменатель

Проекты с нулевой отгрузкой в последнем месяце исключаются из знаменателя — они не формируют базу пролонгации.

> **Методологическое решение:** в знаменатель включаются только проекты с ненулевой отгрузкой в последнем месяце (). Проекты с нулём в последнем месяце технически числятся завершёнными, но реальной базы для пролонгации не формируют — отгружать нечего. Их включение искусственно занизило бы коэффициенты и смазало бы реальную картину работы менеджеров.

In [ ]:
from calculations import calc_all_metrics, calc_annual
from config import EXCLUDE_AM

# Список менеджеров (исключаем строки без А/М — технические записи)
managers  = sorted([am for am in df['AM'].unique() if am != EXCLUDE_AM])

# Порядок строк в таблицах: сначала отдел, потом менеджеры по алфавиту
row_order = ['Отдел в целом'] + managers

# Считаем K1 и K2 для каждого менеджера и отдела за каждый месяц 2023 года
# counts_df — детализация: количество проектов и суммы по каждой группе
results_df, counts_df = calc_all_metrics(df, managers, col_to_idx, month_cols)

# Годовые коэффициенты: взвешенное среднее — сумма числителей / сумма знаменателей
annual_df = calc_annual(results_df)

print('Менеджеры:', managers)
results_df.head(10)

## 3. Сводные таблицы K1 и K2

In [ ]:
import pandas as pd
from calculations import build_pivots

# Объединяем месячные и годовые результаты
all_results = pd.concat([results_df, annual_df], ignore_index=True)

# Сводные таблицы: строки — менеджеры, столбцы — месяцы + год
pivot_k1, pivot_k2, month_order = build_pivots(all_results, row_order)

# Форматирование: числа → проценты, NaN → '—' (нет проектов-кандидатов)
fmt = lambda x: f'{x:.1%}' if str(x) != 'nan' else '—'

print('К1:')
display(pivot_k1.map(fmt))

print('К2:')
display(pivot_k2.map(fmt))

## 4. Годовые коэффициенты

Взвешенное среднее: сумма числителей / сумма знаменателей за все месяцы 2023 года.

In [ ]:
annual_df[['Менеджер', 'k1', 'k2', 'k1_den', 'k1_num', 'k2_den', 'k2_num']].round(3)

## 5. Генерация Excel-отчёта

In [ ]:
from generate_excel import generate_report

generate_report()

# В Google Colab — автоматически скачиваем файл
try:
    from google.colab import files
    files.download('prolongation_report.xlsx')
except ImportError:
    print('Файл сохранён в текущей папке: prolongation_report.xlsx')